# Appendix E, part 2: component-level comparison of one generated signal and the model's output

One signal, one model, two pages. Three switches at the top decide the run: `FAMILY`, either
`"tsmixup"` or `"kernelsynth"`; `MODEL_SOURCE`, either `"retrained"` for the sweep checkpoint of the
geometry under study or `"published"` for the original Chronos-Bolt released by Amazon, which is
itself P = S = 16 and so is the same geometry trained by other people on other data; and
`K_COMPONENTS`, how many components the pages show. Each run writes both pages, named after the
family and the model, so successive runs do not overwrite one another.

* **Page A --- are the components of the input present in the output?** On the left the generated
  signal and, through arrows, its components at a set of reference frequencies: for Light TSMixup
  those are the frequencies the generator actually drew, known by construction; for KernelSynth,
  whose draws have no line spectrum, the strongest lines of the signal itself. On the right the
  model's output fitted at the *same* frequencies, so the two columns are directly comparable.
* **Page B --- where are the components of the output?** The strongest lines of the model's output,
  and the input fitted at those same frequencies. A line that is strong on the right of page B and
  absent on its left is content the model introduced.

Amplitudes are least squares at a known frequency, `pl.fit_amp_phase`, the same estimator the
recovery ratio uses, so nothing here depends on the width of a DFT bin.

**This is descriptive.** It shows frequency information loss; it is not evidence of structural
aliasing, which is what the Bayesian models of the report decide. Everything it depends on lives in
`chronos/bayesian/`.


## 0, Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

# Where the repository is. Locally the notebook sits inside it and nothing has to be set; on
# Colab the working directory is /content, so the search widens to the usual places and, failing
# those, the repository is cloned. Setting REPO_DIR (or the PATCHALIASING_REPO environment
# variable) to the checkout skips the search entirely.
REPO_DIR = None                    # e.g. "/content/drive/MyDrive/patchAliasing"
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
CLONE_IF_MISSING = True            # False to fail with a message instead of cloning

MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def _ok(p) -> bool:
    return p is not None and (Path(p) / MARKER).exists()


def find_repo() -> Path:
    """Locate the checkout: an explicit setting, then the parents, then the usual Colab places."""
    explicit = REPO_DIR or os.environ.get("PATCHALIASING_REPO")
    if explicit:
        if _ok(explicit):
            return Path(explicit).resolve()
        raise FileNotFoundError(f"REPO_DIR is set to {explicit}, but {MARKER} is not under it")

    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:                       # the notebook inside the checkout
        if _ok(cand):
            return cand

    roots = [here, Path("/content"), Path("/content/drive/MyDrive"),
             Path("/content/drive/MyDrive/Colab Notebooks"), Path.home()]
    for root in roots:                                       # a checkout beside the notebook
        if not root.exists():
            continue
        for cand in [root, *(d for d in root.iterdir() if d.is_dir())]:
            if _ok(cand):
                return cand.resolve()

    if not CLONE_IF_MISSING:
        raise FileNotFoundError(
            f"{MARKER} not found. Set REPO_DIR to the checkout, or allow CLONE_IF_MISSING.")

    target = here / "patchAliasing"                          # last resort: fetch it
    if not (target / ".git").exists():
        print(f"cloning {REPO_URL} -> {target}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not _ok(target):
        raise FileNotFoundError(f"{MARKER} missing from {target}")
    return target.resolve()


REPO  = find_repo()
BAYES = REPO / "chronos" / "bayesian"
sys.path.insert(0, str(BAYES))
sys.modules.pop("probe_lib", None)          # so a git pull is picked up without a kernel restart
print("repository:", REPO)

repository: /content/patchAliasing


In [2]:
# Dependencies.  A local machine that has already run the Bayesian notebooks has all of these;
# a fresh Colab runtime has numpy, pandas, matplotlib, scikit-learn and torch, but not the
# Chronos pipeline classes, which come from `chronos-forecasting`.
import importlib.util, subprocess, sys

for module, package in (("torch", "torch"), ("chronos", "chronos-forecasting")):
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import chronos
# The repository has its own top-level `chronos/` directory.  It shadows the library whenever the
# working directory is the repository root, and the error that follows is confusing, so it is
# caught here instead.
if not hasattr(chronos, "BaseChronosPipeline"):
    raise ImportError(
        f"`chronos` resolved to {getattr(chronos, '__file__', '?')}, which is the repository's own "
        "folder rather than the chronos-forecasting library. Run this notebook from a working "
        "directory that is not the repository root.")
print("chronos:", chronos.__file__)

chronos: /usr/local/lib/python3.13/dist-packages/chronos/__init__.py


In [3]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
import probe_lib as pl

SEED = 50
np.random.seed(SEED)
FS, CTX, PRED, BAND = pl.FS, pl.CTX, pl.PRED, pl.BAND

from matplotlib.patches import ConnectionPatch

# ------------------------------------------------------------------------------------- #
#  CONFIGURATION
# ------------------------------------------------------------------------------------- #
FAMILY = "kernelsynth"          # "tsmixup" or "kernelsynth": the signal family

# Which model reconstructs the signal.
#   "retrained": the sweep checkpoint of the geometry below, the one the report analyses
#   "published": the original Chronos-Bolt released by Amazon, untouched by this project
# The published tiny model is itself P = S = 16, so the two are the same geometry trained by
# different people on different data, which is what makes them worth putting side by side.
MODEL_SOURCE = "published"  # "retrained" or "published"
PUBLISHED_ID = "amazon/chronos-bolt-tiny"
REF_P, REF_S = 16, 16       # used when MODEL_SOURCE is "retrained"; the published one carries
                            # its own, and the geometry is read back off whichever is loaded

# Light TSMixup draw. K is the number of components the page shows: raise it for a richer
# mixture, at the cost of one more row of panels. The generator itself picks k ~ U{1, K}, which
# K_RANDOM reproduces; fixed K is the default so the two families give pages of equal height.
K_COMPONENTS = 8
K_RANDOM     = False        # True: k ~ U{1, K}, as the generator does
ALPHA_DIR    = 1.5          # Dirichlet concentration
MIN_SEP_DRAW = 10.0         # Hz between two drawn components
POOL_FMAX    = 150.0        # ceiling applied to the source pool here, whatever the checkout
                            # supplies. A checkout whose probe_lib predates TSMIXUP_POOL_FMAX
                            # still builds the pool to the top of the band, and this keeps the
                            # notebook's pool the intended one either way. None = leave as found
DRAW_FMIN    = None         # narrow the range a single draw may use, inside the pool above.
DRAW_FMAX    = None         # None uses the pool's own range

# How many components each side shows. A Light TSMixup draw holds at most K of them and they are
# all displayed; the prediction is under no such constraint -- it may hold lines the input never
# had -- so it gets its own, larger count. For KernelSynth the input has no components to speak of
# and its strongest lines stand in for them.
N_PEAKS_REAL = 6            # KernelSynth only: lines read off the real signal. A KernelSynth
                            # draw is broadband and carries no drawn frequencies to name, so its
                            # real side is read as lines, as the prediction's is
N_PEAKS_PRED = 8            # lines read off the prediction, on pages B and C
SPECTRUM_BIN_HZ = 1.0       # spacing of the frequency grid the transforms are evaluated on.
                            # Reached by zero padding, which interpolates the spectrum: it places
                            # a peak far more precisely than the raw bin spacing would, but it
                            # does NOT let two nearby lines be told apart -- that stays fs/N,
                            # printed beside it as the resolving power
PEAK_MIN_SEP = 6.0          # Hz between two peaks, so one lobe is not counted twice
PEAK_MIN_REL = 0.05         # and a line below this fraction of the strongest one is not a line

# How much of each signal the least-squares fit sees. The rollout is only faithful near its
# start -- it is fed its own output and drifts -- so the prediction's amplitude and phase are read
# off its first samples rather than off the whole trace. Lines are still *found* on the full
# trace, where the spectrum has resolution; only the fit is windowed.
PRED_CUT        = None      # samples of the prediction kept; None keeps it whole. Everything
                            # downstream -- plots, peak search, fits -- sees only what is kept
MATCH_TIME_SCALE = True     # panels drawn to the same milliseconds-per-inch, so a shorter
                            # prediction comes out as a narrower panel. False makes them equal
REAL_SEGMENT = "future"     # which stretch of the real signal is shown and fitted beside it:
                            #   "future"       the true continuation, the same instants the
                            #                  prediction covers -- the like-for-like comparison
                            #   "context_tail" the last samples the model was given
                            #   "full"         the whole draw
FIT_WINDOW_PRED = 64        # samples used for the fit, clamped to what PRED_CUT kept; None = all
FIT_WINDOW_REAL = None      # the same for the real signal
MIN_CYCLES_WARN = 1.5       # a fit spanning fewer cycles than this is flagged on the panel
MIN_AMP_REL     = 0.45      # a prediction line is dropped unless it reaches this fraction of
                            # the reference amplitude below
MIN_AMP_REF     = "prediction"  # what the fraction is of:
                            # "prediction":     the strongest component fitted in the prediction
                            # "output_signal":  the prediction waveform's own peak amplitude, a
                            #                   harder bar: a line has to account for that much of
                            #                   the signal, not merely beat the other lines
                            # "real":           the strongest component of the real signal
MIN_AMP_ABS     = None      # an absolute floor, when one is wanted, overriding both
COMPONENT_WINDOW = "signal" # time axis of the component panels:
                            #   "signal": the same absolute window as the signal it came from,
                            #             so every panel of the page shares one time axis
                            #   "cycles": a few cycles from zero, easier to read one waveform
COMPONENT_CYCLES = 4        # how many, when COMPONENT_WINDOW is "cycles"
AMP_SCALE       = "real"    # vertical scale of the component panels of figure 3:
                            #   "real": one scale for the whole page, set by the strongest
                            #           component of the real signal, so a weak prediction reads
                            #           as weak instead of being blown up by autoscaling
                            #   "own":  each panel scaled to what it holds
JOINT_FIT       = True      # fit all the frequencies of a page together rather than one at a
                            # time. Over a short window two frequencies inside one bin are nearly
                            # the same basis vector, and separate fits then report the same energy
                            # twice; a joint fit shares it out instead, and says when it cannot

OUT_MODE  = "horizon"       # "horizon": the model's own forecast, PRED samples, which is what a
                            #            deployment actually gets and is far shorter than the context
                            # "rollout": feed the median forecast back until GEN_LEN samples. Finer
                            #            spectra, but the trace is partly the model reading itself
GEN_LEN   = 512             # rollout length, so the output spectrum has 1 Hz bins
BATCH     = 64
DRAW_SEED = SEED            # change this to see a different draw of the same family
INJECT_TONE = None          # Hz, or None: an arbitrary probe tone added to the background

# A component planted on one of the geometry's own lock frequencies, before the signal is given
# to the model. This is the deliberate version of the question the report asks: the input then
# certainly carries energy at a predicted site, and the figures show what comes back.
INJECT_LOCK       = False   # True to plant it
INJECT_LOCK_HZ    = None    # a specific lock frequency, or None to draw one at random
INJECT_LOCK_REL   = 1.25    # its amplitude, as a multiple of the strongest existing component
INJECT_LOCK_IN_POOL = True  # draw only from locks the source pool could also have supplied
INJECT_LOCK_PHASE = "random"   # "random", or a phase in radians

USE_STUB_FORECASTER = False # True only to check the layout without the checkpoints

OUT = BAYES / "_run" / "appendixE"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(DRAW_SEED)

NAME = {"tsmixup": "Light TSMixup", "kernelsynth": "KernelSynth"}[FAMILY]
INJECTED_LOCK = None        # the planted component, set in section 1 when INJECT_LOCK is on


def run_suffix():
    """The tag every file of this run carries: family, model, and the planted component.

    It is a function and not a constant, and it reads the settings as they stand when it is
    called. The geometry it names is therefore the one actually loaded in section 2 rather than
    the one requested here, a run on the published checkpoint always says `published`, and a run
    that plants a component says where. Two runs that differ in any of those write two different
    names, so a page can never be quietly left over from an earlier one.
    """
    if USE_STUB_FORECASTER:
        src = "stub"
    elif MODEL_SOURCE == "published":
        src = "published"
    else:
        src = f"p{REF_P}-s{REF_S}"
    tag = f"{FAMILY}_{src}"
    if INJECTED_LOCK:
        tag += "_lock" + f"{INJECTED_LOCK:g}".replace(".", "p")
    return tag


print(f"family={NAME}  model={MODEL_SOURCE}  K={K_COMPONENTS}"
      f"{' (random)' if K_RANDOM else ''}  output='{OUT_MODE}'  seed={DRAW_SEED}")
print(f"fit window: prediction {FIT_WINDOW_PRED or 'all'} samples, "
      f"real {FIT_WINDOW_REAL or 'all'} samples")
print(f"components shown: real {'K = %d' % K_COMPONENTS if FAMILY == 'tsmixup' else 'up to %d lines' % N_PEAKS_REAL}, "
      f"prediction up to {N_PEAKS_PRED} lines")
print(f"figures -> {OUT / 'figures'}")
print(f"files of this run are named  *_{run_suffix()}.*  "
      f"(the model part is settled in section 2, by the checkpoint that actually loads)")

family=KernelSynth  model=published  K=8  output='horizon'  seed=50
fit window: prediction 64 samples, real all samples
components shown: real up to 6 lines, prediction up to 8 lines
figures -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures
files of this run are named  *_kernelsynth_published.*  (the model part is settled in section 2, by the checkpoint that actually loads)


## 1, The signal

In [4]:
# The source pool of Light TSMixup. `probe_lib.tsmixup_pool` is used when the checkout has it;
# a checkout that predates it gets the same construction rebuilt here, from primitives both
# versions carry. The four groups are those of Appendix B: every integer of the band, the
# non-integer members of F_lock, the control frequencies of every candidate, and 150 further
# non-integer frequencies held clear of both.
POOL_N_FREE, POOL_GUARD = 150, 0.2


def _pool_rebuilt(n_free=POOL_N_FREE, guard=POOL_GUARD, models=None):
    models = models or pl.MODELS
    lo = pl.BAND[0]
    # the pool's own ceiling when the checkout defines one, the analysed band otherwise
    hi = min(pl.BAND[1], float(getattr(pl, "TSMIXUP_POOL_FMAX", pl.BAND[1])),
             pl.BAND[1] if POOL_FMAX is None else float(POOL_FMAX))
    lock = set()
    for P, S in models:
        lock |= {round(f, 9) for f in pl.f_lock(P, S, fmax=hi, fmin=lo)}
    lock_sorted = sorted(lock)
    integers = [float(f) for f in range(int(np.ceil(lo)), int(np.floor(hi)) + 1)]
    non_integer_locks = [f for f in lock_sorted if abs(f - round(f)) > 1e-9]
    controls = set()
    for P, S in models:
        for fk in pl.f_lock(P, S, fmax=hi, fmin=lo):
            d = pl.control_offset(P, S, fk)
            if not np.isfinite(d):
                continue
            for f in (fk - d, fk + d):
                if lo <= f <= hi:
                    controls.add(round(f, 9))

    def clear(f):
        return (min(abs(f - l) for l in lock_sorted) >= guard) and (abs(f - round(f)) >= guard)

    free = []
    for f0 in np.linspace(lo + 0.5, hi - 0.5, n_free):
        f, j = float(f0), 0
        while not clear(f) and j < 40:                     # nudge until it is clear of both grids
            j += 1
            f = float(f0) + 0.05 * ((j + 1) // 2) * (1 if j % 2 else -1)
        free.append(round(f, 4))
    return sorted(set(integers) | {round(f, 9) for f in non_integer_locks} | controls | set(free))


_POOL_CACHE = {}


def source_pool(verbose=True):
    """The Light TSMixup pool, from probe_lib when it has one, rebuilt here otherwise."""
    if "pool" not in _POOL_CACHE:
        fn = getattr(pl, "tsmixup_pool", None)
        if fn is None:
            print("note: this checkout of probe_lib has no tsmixup_pool; the pool is rebuilt "
                  "in-notebook. Push the updated probe_lib.py to keep the two from drifting.")
            _POOL_CACHE["pool"] = _pool_rebuilt()
            _POOL_CACHE["source"] = "rebuilt in-notebook"
        else:
            _POOL_CACHE["pool"] = fn()
            _POOL_CACHE["source"] = "probe_lib.tsmixup_pool"
    pool = _POOL_CACHE["pool"]
    if POOL_FMAX is not None:                    # hold the ceiling whatever the checkout gave
        pool = [f for f in pool if f <= float(POOL_FMAX)]
    if verbose and not _POOL_CACHE.get("announced"):
        _POOL_CACHE["announced"] = True
        lib_cap = getattr(pl, "TSMIXUP_POOL_FMAX", None)
        where = ("probe_lib defines the cap" if lib_cap is not None
                 else "this checkout's probe_lib has no cap; applied here")
        print(f"pool: {len(pool)} frequencies from {min(pool):g} to {max(pool):g} Hz "
              f"({_POOL_CACHE['source']}; {where})")
    return pool


def light_tsmixup_draw(rng, k=None, alpha=None, length=pl.CANON_LEN, min_sep=None, pool=None):
    """One Light TSMixup realisation, with the frequencies and weights of the draw returned.

    Algorithm 1 of the appendix: one sinusoid per drawn pool frequency, each divided by its mean
    absolute value, combined under symmetric Dirichlet weights, normalised to unit variance.

    The settings are read from the configuration cell at call time, not bound as defaults: a
    default would freeze the value this cell last ran with, so editing K and re-running only the
    configuration would silently keep the old number of components.
    """
    k = K_COMPONENTS if k is None else int(k)
    alpha = ALPHA_DIR if alpha is None else float(alpha)
    min_sep = MIN_SEP_DRAW if min_sep is None else float(min_sep)
    pool = np.asarray(source_pool() if pool is None else pool, dtype=float)
    lo = min(pool) if DRAW_FMIN is None else float(DRAW_FMIN)
    hi = max(pool) if DRAW_FMAX is None else float(DRAW_FMAX)
    full = len(pool)
    pool = pool[(pool >= lo) & (pool <= hi)]
    if len(pool) < full:
        print(f"draw restricted to [{lo:g}, {hi:g}] Hz: {len(pool)} of {full} pool frequencies")
    if len(pool) < k:
        raise ValueError(f"only {len(pool)} frequencies in [{lo:g}, {hi:g}] Hz, K = {k}")
    if K_RANDOM:
        k = int(rng.integers(1, k + 1))              # the generator's own k ~ U{1, K}
    freqs = []
    while len(freqs) < k:                       # redraw a frequency that would overlap another
        f = float(rng.choice(pool))
        if all(abs(f - g) >= min_sep for g in freqs):
            freqs.append(f)
    comps  = [np.asarray(pl.make_tone(f, 0.0, length, 1.0), dtype=float) for f in freqs]
    scaled = [c / np.mean(np.abs(c)) for c in comps]
    w = rng.dirichlet(np.full(k, alpha))
    x = np.sum([wi * si for wi, si in zip(w, scaled)], axis=0)
    return (x / x.std()).astype(np.float32), sorted(freqs), w


def spectrum(x, fs=FS, window=True, bin_hz=None):
    """One-sided magnitude spectrum on a grid of `bin_hz`, scaled so a tone of amplitude A reads A.

    The grid is reached by zero padding. That interpolates between the true bins and lets a peak
    be placed to a fraction of them; it does not add resolving power, which stays fs/N.
    """
    bin_hz = SPECTRUM_BIN_HZ if bin_hz is None else float(bin_hz)
    y = np.asarray(x, float); y = y - y.mean()
    w = np.hanning(len(y)) if window else np.ones(len(y))
    nfft = max(len(y), int(round(fs / max(bin_hz, 1e-9))))
    mag = np.abs(np.fft.rfft(y * w, n=nfft)) * 2.0 / np.sum(w)
    return np.fft.rfftfreq(nfft, d=1 / fs), mag


def locks_of(P, S):
    """The predicted sites of one geometry: F_lock = {k fs/P} union {c fs/S}."""
    return sorted({round(f, 3) for f in pl.patch_nulls(P)} |
                  {round(f, 3) for f in pl.stride_locks(S)})


def is_lock(f, tol=1.0):
    """Whether `f` sits on the comb of the geometry currently selected."""
    return any(abs(f - l) <= tol for l in locks_of(REF_P, REF_S))


def pool_limit():
    """The highest frequency the input could possibly contain, for this family and this draw."""
    if FAMILY != "tsmixup":
        return None
    cap = float(getattr(pl, "TSMIXUP_POOL_FMAX", BAND[1]))
    if POOL_FMAX is not None:
        cap = min(cap, float(POOL_FMAX))
    if DRAW_FMAX is not None:
        cap = min(cap, float(DRAW_FMAX))
    return min(cap, BAND[1])


def above_pool(f):
    """Whether `f` is above anything the generator could have put in: the model's own content."""
    cap = pool_limit()
    return cap is not None and f > cap + 1e-6


def top_peaks(x, n_peaks=None, min_sep=None, band=BAND, fs=FS, min_rel=None):
    """The strongest spectral lines of `x` inside `band`.

    At most `n_peaks`, no two closer than `min_sep`, and none weaker than `min_rel` of the
    strongest: a signal with three lines returns three, not eight rows of noise.
    """
    n_peaks = N_PEAKS_PRED if n_peaks is None else int(n_peaks)
    # never below the resolving power: two "peaks" closer than fs/N are one line seen twice,
    # whatever grid the transform is evaluated on
    min_sep = max(PEAK_MIN_SEP, fs / len(np.asarray(x))) if min_sep is None else float(min_sep)
    min_rel = PEAK_MIN_REL if min_rel is None else float(min_rel)
    fr, mag = spectrum(x, fs=fs)
    m = (fr >= band[0]) & (fr <= band[1])
    fr, mag = fr[m], mag[m]
    if not len(mag) or mag.max() <= 0:
        return []
    floor = min_rel * mag.max()
    picked = []
    for i in np.argsort(mag)[::-1]:
        if mag[i] < floor:
            break
        f = float(fr[i])
        if all(abs(f - p) >= min_sep for p in picked):
            picked.append(f)
        if len(picked) >= n_peaks:
            break
    return sorted(picked)


def kernelsynth_draw(seed, length=pl.CANON_LEN):
    """One KernelSynth realisation, with the kernels of the draw recorded.

    `probe_lib.background` does exactly this and hands back the array alone; the generator is
    called here with the same parameters and the same seed, so the signal is the one probe_lib
    would have produced and `last_kernels` can be read off it. `signalGenerator` is importable
    because probe_lib puts the generator directory on the path when it is imported.
    """
    import signalGenerator as sg
    tmp = BAYES / "_gen_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    params = {"J": 5, "l_syn": int(length), "fs": FS, "jitter": 1e-4, "P": 16}
    gen = sg.runKernelSynth(params, int(seed), tmp)
    x = np.asarray(gen.generate(), float).ravel()[:int(length)]
    sd = x.std()
    x = (x / sd) if sd > 1e-8 else x                  # unit variance, as probe_lib does
    return x.astype(np.float32), list(getattr(gen, "last_kernels", [])), dict(gen.getParameters())


def amp_of(x, f):
    """Least-squares amplitude of `x` at `f`, before the decomposition cell defines its own."""
    y = np.asarray(x, float)
    return float(pl.fit_amp_phase(y, np.arange(len(y)) / FS, f)[0])


def plant_lock_component(x, rng):
    """Add a component on one of the geometry's lock frequencies, and return it with the choice.

    The amplitude is set relative to the strongest component the signal already has, so the
    planted tone is the dominant one by construction and cannot be missed for want of energy.
    The result is renormalised to unit variance, as every background here is, which scales the
    whole mixture and leaves the ratio between components untouched.
    """
    locks = [f for f in locks_of(REF_P, REF_S) if BAND[0] <= f <= BAND[1]]
    cap = pool_limit()
    if INJECT_LOCK_IN_POOL and cap is not None:
        locks = [f for f in locks if f <= cap] or locks
    if not locks:
        raise ValueError(f"p{REF_P}-s{REF_S} has no lock frequency in {BAND}")
    f = float(INJECT_LOCK_HZ) if INJECT_LOCK_HZ is not None else float(rng.choice(locks))
    base = max([amp_of(x, g) for g in REF_FREQS] + [1e-9])
    amp = INJECT_LOCK_REL * base
    phase = (float(rng.uniform(0, 2 * np.pi)) if INJECT_LOCK_PHASE == "random"
             else float(INJECT_LOCK_PHASE))
    y = np.asarray(x, float) + np.asarray(pl.make_tone(f, phase, len(x), amp), float)
    return (y / y.std()).astype(np.float32), f, amp, phase


def build_signal(family=None, seed=None):
    """Draw one realisation and bind everything the later sections read off it.

    This section is a function rather than a run of statements so that a second family costs one
    call and not a second notebook: section 6 runs it again for the other generator, and what is
    bound here is exactly what figures 3 and 4 read.
    """
    global FAMILY, NAME, RNG, DRAW_SEED, SIGNAL, CONTEXT, REF_FREQS, REF_SOURCE, WEIGHTS
    global KERNELS, KS_PARAMS, INJECTED_LOCK
    if family is not None:
        FAMILY = str(family)
    if seed is not None:
        DRAW_SEED = int(seed)
    NAME = {"tsmixup": "Light TSMixup", "kernelsynth": "KernelSynth"}[FAMILY]
    RNG = np.random.default_rng(DRAW_SEED)
    KERNELS, KS_PARAMS, WEIGHTS, INJECTED_LOCK = [], {}, None, None
    print(f"--- {NAME}, seed {DRAW_SEED} " + "-" * 40)

    if FAMILY == "tsmixup":
        SIGNAL, REF_FREQS, WEIGHTS = light_tsmixup_draw(RNG)
        REF_SOURCE = "the frequencies drawn by the generator"
        print(f"drew {len(REF_FREQS)} components (K = {K_COMPONENTS}"
              f"{', random' if K_RANDOM else ''})")
        print("drawn components [Hz]:", [round(f, 3) for f in REF_FREQS])
        print("mixing weights      :", np.round(WEIGHTS, 3))
        hi_used = pool_limit() or max(source_pool(verbose=False))
        assert max(REF_FREQS) <= hi_used + 1e-6, (
            f"drew {max(REF_FREQS):g} Hz above the {hi_used:g} Hz limit: the pool or the cell "
            "defining light_tsmixup_draw is stale, re-run it")
        if not K_RANDOM:
            assert len(REF_FREQS) == K_COMPONENTS, (
                f"{len(REF_FREQS)} components for K = {K_COMPONENTS}: the cell defining "
                "light_tsmixup_draw is stale, re-run it")
    else:
        SIGNAL, KERNELS, KS_PARAMS = kernelsynth_draw(DRAW_SEED)
        REF_FREQS, WEIGHTS = top_peaks(SIGNAL, n_peaks=N_PEAKS_REAL), None
        REF_SOURCE = f"the {len(REF_FREQS)} strongest lines of the generated signal"
        print("kernels drawn      :", " ".join(KERNELS))
        print("generator settings :", {k: v for k, v in KS_PARAMS.items() if k != "inject"})
        print("strongest lines [Hz]:", [round(f, 2) for f in REF_FREQS])

    if INJECT_TONE is not None:
        SIGNAL = (SIGNAL + pl.make_tone(INJECT_TONE, 0.0, len(SIGNAL), pl.TONE_SNR)).astype(np.float32)
        print(f"probe tone injected at {INJECT_TONE} Hz")

    if INJECT_LOCK:
        SIGNAL, INJECTED_LOCK, _amp, _ph = plant_lock_component(SIGNAL, RNG)
        REF_FREQS = sorted(set(list(REF_FREQS) + [INJECTED_LOCK]))
        print(f"planted a component at {INJECTED_LOCK:g} Hz, a lock of p{REF_P}-s{REF_S}: "
              f"amplitude {INJECT_LOCK_REL:g}x the strongest existing component, "
              f"phase {_ph:.2f} rad")
        print("   after renormalising, the components are: "
              + ", ".join(f"{f:g} Hz A={amp_of(SIGNAL, f):.3f}"
                          + ("  <- planted" if f == INJECTED_LOCK else "") for f in REF_FREQS))

    CONTEXT = SIGNAL[:CTX]
    print(f"signal {len(SIGNAL)} samples, context {len(CONTEXT)}, std {SIGNAL.std():.3f}")
    return SIGNAL


build_signal()

--- KernelSynth, seed 50 ----------------------------------------
kernels drawn      : RBF * Linear + RQ * RQ
generator settings : {'generator': 'KernelSynth', 'J': 5, 'l_syn': 544, 'fs': 512, 'jitter': 0.0001, 'P': 16}
strongest lines [Hz]: [2.82]
signal 544 samples, context 480, std 1.000


array([-18.323122, -18.360756, -18.349016, -18.3546  , -18.34822 ,
       -18.357399, -18.354464, -18.37986 , -18.361782, -18.38419 ,
       -18.394205, -18.390352, -18.397837, -18.412706, -18.419996,
       -18.406862, -18.433441, -18.433708, -18.45661 , -18.446964,
       -18.466568, -18.470123, -18.4825  , -18.470776, -18.48027 ,
       -18.477795, -18.485477, -18.505856, -18.518972, -18.525633,
       -18.508947, -18.516098, -18.528858, -18.531874, -18.536966,
       -18.542501, -18.550785, -18.55077 , -18.548687, -18.532133,
       -18.564358, -18.54316 , -18.559246, -18.565641, -18.553707,
       -18.570374, -18.558125, -18.591394, -18.57962 , -18.585497,
       -18.585186, -18.584759, -18.570509, -18.566689, -18.582008,
       -18.587183, -18.577404, -18.568335, -18.572678, -18.563267,
       -18.560474, -18.548956, -18.5796  , -18.571407, -18.57104 ,
       -18.554825, -18.550129, -18.54645 , -18.52951 , -18.518572,
       -18.546247, -18.524757, -18.51394 , -18.507444, -18.500

In [5]:
# What produced this signal, in full: a figure is only as good as the draw behind it, and both
# families are reproducible from what is printed here plus DRAW_SEED.


def describe_draw(show=True):
    """The parameters of the draw currently in memory, printed, tabulated and archived."""
    global PARAMS, SETTINGS
    if FAMILY == "tsmixup":
        drawn = [f for f in REF_FREQS if f != INJECTED_LOCK]
        w_of = dict(zip(sorted(drawn), [WEIGHTS[i] for i in np.argsort(np.argsort(drawn))]))
        PARAMS = pd.DataFrame({
            "component": np.arange(1, len(REF_FREQS) + 1),
            "frequency_hz": REF_FREQS,
            "weight": [w_of.get(f, np.nan) for f in REF_FREQS],
            "planted": [f == INJECTED_LOCK for f in REF_FREQS],
            "in_F_lock": [is_lock(f) for f in REF_FREQS],
            "cycles_per_patch": [f * REF_P / FS for f in REF_FREQS],
        })
        SETTINGS = {"generator": "Light TSMixup", "K": K_COMPONENTS, "k_random": K_RANDOM,
                    "planted_lock_hz": INJECTED_LOCK,
                    "alpha": ALPHA_DIR, "pool_size": len(source_pool()),
                    "min_separation_hz": MIN_SEP_DRAW,
                    "draw_range_hz": (BAND[0] if DRAW_FMIN is None else DRAW_FMIN,
                                      BAND[1] if DRAW_FMAX is None else DRAW_FMAX),
                    "length": len(SIGNAL),
                    "fs_hz": FS, "seed": DRAW_SEED}
    else:
        PARAMS = pd.DataFrame({"kernel": KERNELS})
        SETTINGS = {"generator": "KernelSynth", **{k: v for k, v in KS_PARAMS.items()
                                                   if k != "inject"},
                    "n_kernels_drawn": len(KERNELS), "planted_lock_hz": INJECTED_LOCK,
                    "seed": DRAW_SEED}

    print("settings:")
    for k, v in SETTINGS.items():
        print(f"    {k:>18} = {v}")
    PARAMS.to_csv(OUT / f"generator_params_{run_suffix()}.csv", index=False)
    if show:
        display(PARAMS.round(4))
    return PARAMS


describe_draw()

settings:
             generator = KernelSynth
                     J = 5
                 l_syn = 544
                    fs = 512
                jitter = 0.0001
                     P = 16
       n_kernels_drawn = 4
       planted_lock_hz = None
                  seed = 50


,kernel
0,RBF
1,* Linear
2,+ RQ
3,* RQ


,kernel
0,RBF
1,* Linear
2,+ RQ
3,* RQ


## 2, What the model returns

In [6]:
class StubProbe:
    """A stand-in for `pl.Probe` that needs no checkpoint: layout checks only.

    It returns a smoothed continuation of the context, which is not a forecast and must never be
    read as one. Every figure produced while `USE_STUB_FORECASTER` is true carries a stamp.
    """
    def __init__(self, P, S):
        self.P, self.S = P, S
        self.tag, self.label = pl.model_tag(P, S), f"stub p{P}-s{S}"
        self.stages = ["output_head"]

    def forecast(self, contexts):
        c = np.asarray(contexts, dtype=np.float32)
        k = np.ones(9) / 9.0
        sm = np.stack([np.convolve(row, k, mode="same") for row in c])
        return np.repeat(sm[:, -1:], PRED, axis=1) * 0.6 + sm[:, -PRED:] * 0.4

    def capture_reg(self, contexts, pipe=None):
        c = np.asarray(contexts, dtype=np.float32)
        return {"output_head": np.stack([c[:, :16], c[:, -16:]], axis=1).reshape(len(c), -1)}

    def close(self):
        pass


def open_probe(P, S, batch_size=64):
    """The real probe, or the stub when the checkpoints are not available."""
    return StubProbe(P, S) if USE_STUB_FORECASTER else pl.Probe(P, S, batch_size=batch_size)


def stamp_stub(fig):
    if USE_STUB_FORECASTER:
        fig.text(0.5, 0.5, "STUB FORECASTER\nNOT A MEASUREMENT", fontsize=42, color="red",
                 alpha=0.16, ha="center", va="center", rotation=30, zorder=99)

In [7]:
def chronos_generate(probe, context, mode=OUT_MODE, gen_len=GEN_LEN):
    """The model's output for one context: the raw horizon, or the fed-back rollout.

    The rollout is the procedure the reconstruction figures use: the median forecast is appended to
    the context and the model re-invoked until `gen_len` samples exist. It is imposed from outside
    and is not Chronos-Bolt's own generation mode, which emits its whole horizon in one step; it is
    used here because 64 samples give 8 Hz bins, too coarse to place a line.
    """
    ctx_len = len(context)
    ctx = np.asarray(context, dtype=np.float32)[None, :]
    if mode == "horizon":
        return probe.forecast(ctx)[0].astype(float)
    gen = np.zeros((1, 0), dtype=np.float32)
    while gen.shape[1] < gen_len:
        step = probe.forecast(ctx)
        gen = np.concatenate([gen, step], axis=1)
        ctx = np.concatenate([ctx, step], axis=1)[:, -ctx_len:]
    return gen[0, :gen_len].astype(float)


class PublishedProbe:
    """The original Chronos-Bolt checkpoint, wrapped in the little of `pl.Probe` used here.

    `pl.Probe` resolves a geometry to one of the project's retrained checkpoints, so it cannot
    load the released model. Only `forecast` is needed, and the geometry is read back off the
    loaded config rather than assumed, so the lock markers are always the model's own.
    """
    def __init__(self, hf_id=PUBLISHED_ID, device=None, batch_size=64):
        import torch
        from chronos import BaseChronosPipeline
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.pipe = BaseChronosPipeline.from_pretrained(hf_id, device_map=self.device)
        cfg = self.pipe.model.config.chronos_config
        self.P = int(cfg["input_patch_size"])
        self.S = int(cfg["input_patch_stride"])
        quantiles = list(cfg["quantiles"])
        self.qi = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles) // 2
        self.tag = f"{pl.model_tag(self.P, self.S)}-published"
        self.label = f"{hf_id}  (P={self.P}, S={self.S})"

    def forecast(self, contexts):
        torch = self.torch
        X = np.asarray(contexts, np.float32)
        if X.ndim == 1:
            X = X[None, :]
        out = []
        with torch.no_grad():
            for i in range(0, len(X), self.batch_size):
                xb = torch.tensor(X[i:i + self.batch_size], device=self.device)
                out.append(self.pipe.predict(xb, prediction_length=PRED)[:, self.qi, :]
                           .float().cpu().numpy())
        return np.concatenate(out, axis=0)

    def close(self):
        del self.pipe
        self.pipe = None
        if self.device == "cuda":
            self.torch.cuda.empty_cache()


def open_model():
    """The model this run uses: the stub, the released checkpoint, or the retrained geometry."""
    if USE_STUB_FORECASTER:
        return StubProbe(REF_P, REF_S)
    if MODEL_SOURCE == "published":
        return PublishedProbe(PUBLISHED_ID, batch_size=BATCH)
    if MODEL_SOURCE != "retrained":
        raise ValueError(f"MODEL_SOURCE must be 'retrained' or 'published', not {MODEL_SOURCE!r}")
    return pl.Probe(REF_P, REF_S, batch_size=BATCH)


def real_segment(kind=None, n=None):
    """The stretch of the real signal put beside the prediction, and where it starts.

    The prediction covers `n` samples immediately after the context, so the continuation over
    exactly those instants is the comparison of record: same instants, same length, same
    frequency resolution. The other two settings are there for inspection.
    """
    kind = REAL_SEGMENT if kind is None else kind
    n = len(OUTPUT) if n is None else int(n)
    if kind == "future":
        return np.asarray(SIGNAL[CTX:CTX + n], float), CTX
    if kind == "context_tail":
        return np.asarray(SIGNAL[CTX - n:CTX], float), CTX - n
    if kind == "full":
        return np.asarray(SIGNAL, float), 0
    raise ValueError(f"REAL_SEGMENT must be 'future', 'context_tail' or 'full', not {kind!r}")


def build_output():
    """Load the model, run it on the current context, and bind the prediction beside its real twin.

    A function for the same reason section 1 is one: the second family of section 6 needs the
    whole of it again, and a copy of these lines would be a second place for them to drift.
    """
    global OUTPUT, REAL, REAL_T0, MODEL_LABEL, REF_P, REF_S

    probe = open_model()
    try:
        print("model:", probe.label)
        if INJECTED_LOCK is not None and (probe.P, probe.S) != (REF_P, REF_S):
            print(f"WARNING: the lock was planted for p{REF_P}-s{REF_S} but the checkpoint is "
                  f"p{probe.P}-s{probe.S}; {INJECTED_LOCK:g} Hz may not be a lock of it")
        REF_P, REF_S = probe.P, probe.S      # the loaded geometry decides where the locks are drawn
        MODEL_LABEL = ("stub forecaster, not a model" if USE_STUB_FORECASTER else
                       f"{PUBLISHED_ID}, P={REF_P}, S={REF_S}" if MODEL_SOURCE == "published"
                       else f"retrained p{REF_P}-s{REF_S}")
        print(f"    every file written from here on is named  *_{run_suffix()}.*")
        if USE_STUB_FORECASTER:
            print("    NOTE: the stub forecaster is in use, so the name says 'stub' and not "
                  f"'{MODEL_SOURCE}'; these pages show the layout, not the model")
        # A rollout is pointless when only the first PRED samples are kept: the leading PRED samples
        # of a rollout are the one-shot forecast, so the loop would be thrown away.
        mode = "horizon" if (PRED_CUT is not None and PRED_CUT <= PRED) else OUT_MODE
        if mode != OUT_MODE:
            print(f"PRED_CUT={PRED_CUT} <= horizon {PRED}: taking the single forecast, no rollout")
        OUTPUT = chronos_generate(probe, CONTEXT, mode=mode)
    finally:
        probe.close()

    if PRED_CUT is not None:
        OUTPUT = OUTPUT[:int(PRED_CUT)]
    REAL, REAL_T0 = real_segment()

    print(f"real segment: '{REAL_SEGMENT}', {len(REAL)} samples starting at "
          f"{REAL_T0 / FS * 1000:.0f} ms")
    print(f"context {len(CONTEXT)} samples ({len(CONTEXT) / FS * 1000:.0f} ms), "
          f"prediction {len(OUTPUT)} samples ({len(OUTPUT) / FS * 1000:.0f} ms), "
          f"std {OUTPUT.std():.4f}")
    print(f"spectra evaluated on a {SPECTRUM_BIN_HZ:g} Hz grid (zero padded)")
    print(f"resolving power: prediction {FS / len(OUTPUT):.1f} Hz, real segment {FS / len(REAL):.1f} Hz"
          f" -- two lines closer than that are one line, not two")
    return OUTPUT


build_output()

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

model: amazon/chronos-bolt-tiny  (P=16, S=16)
    every file written from here on is named  *_kernelsynth_published.*
real segment: 'future', 64 samples starting at 938 ms
context 480 samples (938 ms), prediction 64 samples (125 ms), std 0.2477
spectra evaluated on a 1 Hz grid (zero padded)
resolving power: prediction 8.0 Hz, real segment 8.0 Hz -- two lines closer than that are one line, not two


array([-20.33933067, -20.323843  , -20.31957054, -20.31767845,
       -20.30136871, -20.29535484, -20.28552055, -20.28125381,
       -20.26949883, -20.26602745, -20.25769043, -20.25073624,
       -20.23570824, -20.22655678, -20.21681786, -20.21057892,
       -20.19425583, -20.18315697, -20.1675148 , -20.15741348,
       -20.14451218, -20.13192749, -20.11491013, -20.10709763,
       -20.09882736, -20.08366776, -20.0622654 , -20.04535675,
       -20.02337646, -19.99848938, -19.9782505 , -19.9673233 ,
       -19.9552803 , -19.94399261, -19.92922592, -19.91626167,
       -19.90206146, -19.89008713, -19.87594032, -19.86852837,
       -19.85380554, -19.83547783, -19.81579399, -19.80484581,
       -19.7998848 , -19.79446793, -19.76888275, -19.75335693,
       -19.7463398 , -19.73874855, -19.72315788, -19.71786499,
       -19.69484711, -19.67492294, -19.65407753, -19.63925934,
       -19.61681747, -19.60108948, -19.58787155, -19.57971764,
       -19.56547737, -19.55846596, -19.55469513, -19.55

## 3, Decomposition: fitting the components

In [8]:
C_IN, C_OUT = "#1f4e79", "#c81e3c"
C_PLANT = "#7d3c98"                         # the planted component, wherever it turns up
LOCKS_REF = locks_of(REF_P, REF_S)          # the loaded geometry, so the markers are its own


def is_planted(f, tol=1e-9):
    """Whether `f` is the component planted on a lock frequency before the model saw the signal.

    On the real side the match is exact: that frequency is the one that was added. On the
    prediction side one bin is passed in as the tolerance, because the transform cannot place a
    line more finely than that, so a returned line within a bin of the planted site is that site.
    """
    return INJECTED_LOCK is not None and abs(f - INJECTED_LOCK) <= tol


def report_written(p):
    """Name, size and time of every file written, and the directory it went to.

    A page is read back, or copied to Drive, by name, and a name says nothing about when it was
    made: a run that stops halfway leaves the previous file in place and it looks exactly like a
    fresh one. The time stamp printed here is what tells the two apart.
    """
    import time
    st = p.stat()
    print(f"    wrote {p.name}  {st.st_size / 1024:.0f} kB  "
          f"{time.strftime('%H:%M:%S', time.localtime(st.st_mtime))}  -> {p.parent}")


def component_at(x, f, fs=FS, n=None):
    """Least-squares amplitude and phase of `x` at `f` Hz, the estimator R is fitted with.

    `n` truncates the fit to the leading `n` samples. A window shorter than a cycle of `f` cannot
    separate amplitude from phase, so `cycles_in` is reported beside the fit and flagged on the
    panel when it falls below `MIN_CYCLES_WARN`.
    """
    y = np.asarray(x, float)
    if n is not None:
        y = y[:int(min(int(n), len(y)))]
    t = np.arange(len(y)) / fs
    amp, ph = pl.fit_amp_phase(y, t, f)
    return float(amp), float(ph)


def components_of(x, freqs, n=None, joint=None, fs=FS):
    """Amplitude and phase of `x` at each of `freqs`, as a list of (amp, phase).

    With `joint`, one design matrix carries every frequency at once, so energy shared by two
    frequencies closer than the window can resolve is split between them instead of being
    reported in full for each. An ill-conditioned design means exactly that: the window cannot
    tell those frequencies apart, and the notebook says so rather than returning a tidy number.
    """
    joint = JOINT_FIT if joint is None else joint
    y = np.asarray(x, float)
    if n is not None:
        y = y[:int(min(int(n), len(y)))]
    t = np.arange(len(y)) / fs
    if not joint or len(freqs) < 2:
        return [component_at(y, f, fs=fs) for f in freqs]
    cols = []
    for f in freqs:
        cols += [np.cos(2 * np.pi * f * t), np.sin(2 * np.pi * f * t)]
    X = np.stack(cols + [np.ones_like(t)], axis=1)
    cond = np.linalg.cond(X)
    if cond > 1e3:
        print(f"    joint fit poorly conditioned (cond = {cond:.0f}): "
              f"{FS / len(y):.1f} Hz bins cannot separate these frequencies")
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return [(float(np.hypot(beta[2 * i], beta[2 * i + 1])),
             float(np.arctan2(beta[2 * i + 1], beta[2 * i]))) for i in range(len(freqs))]


def reference_amplitude(pred_amps=None):
    """The amplitude the threshold is a fraction of.

    With MIN_AMP_REF = "prediction" it is the prediction's own strongest component, so what is
    kept is what actually builds the predicted signal and a quiet prediction is judged on its own
    scale rather than against a loud input.
    """
    if MIN_AMP_REF == "prediction":
        return max(pred_amps) if pred_amps else 1.0
    if MIN_AMP_REF == "output_signal":
        y = np.asarray(OUTPUT, float)
        return float(np.max(np.abs(y - y.mean()))) or 1.0
    if MIN_AMP_REF == "real":
        amps = [a for a, _ in components_of(REAL, REF_FREQS, n=FIT_WINDOW_REAL)]
        return max(amps) if amps else 1.0
    raise ValueError("MIN_AMP_REF must be 'prediction', 'output_signal' or 'real', "
                     f"not {MIN_AMP_REF!r}")


def amplitude_floor(pred_amps=None):
    """The absolute amplitude a prediction line has to reach to be shown."""
    if MIN_AMP_ABS is not None:
        return float(MIN_AMP_ABS)
    return MIN_AMP_REL * reference_amplitude(pred_amps)


def prediction_lines(n_peaks=None, min_amp=None, verbose=True):
    """The lines of the prediction worth showing: peaks found, then fitted, then thresholded.

    The threshold is applied to the fit of the set that is finally drawn, not to a fit of the
    candidates. With a joint fit those differ: dropping a line changes the amplitudes of the ones
    that remain, so a single pass can leave a line on the page whose displayed amplitude is below
    the rule. Iterating to a fixed point removes that inconsistency, and what the page shows then
    satisfies the rule as printed.
    """
    cand = top_peaks(OUTPUT, n_peaks=n_peaks)
    if not cand:
        return []
    keep, amps, floor = list(cand), [], 0.0
    for _ in range(8):
        amps = [a for a, _ in components_of(OUTPUT, keep, n=FIT_WINDOW_PRED)]
        floor = amplitude_floor(amps) if min_amp is None else float(min_amp)
        nxt = [f for f, a in zip(keep, amps) if a >= floor]
        if not nxt:                                   # never return nothing: keep the strongest
            nxt = [keep[int(np.argmax(amps))]]
        if nxt == keep:
            break
        keep = nxt
    if verbose:
        dropped = len(cand) - len(keep)
        if dropped:
            print(f"    {dropped} prediction line(s) below A = {floor:.4f} "
                  f"({MIN_AMP_REL:g} x the strongest {MIN_AMP_REF} component) dropped")
        kept = ", ".join(f"{f:.2f} Hz (A={a:.3f})" for f, a in zip(keep, amps) if f in keep)
        print(f"    kept: {kept}")
    return keep


def resolution_hz(x, fs=FS):
    """Width of one bin: two frequencies closer than this are one line, not two."""
    return fs / len(x)


def near_input(f, tol=None):
    """The input component `f` is within one bin of, if any: a line, not new content."""
    tol = resolution_hz(OUTPUT) if tol is None else tol
    if not len(REF_FREQS):
        return None
    j = int(np.argmin([abs(f - g) for g in REF_FREQS]))
    return REF_FREQS[j] if abs(f - REF_FREQS[j]) <= tol else None


def cycles_in(f, n, fs=FS, total=None):
    """How many cycles of `f` a fit window of `n` samples spans."""
    n = total if n is None else min(int(n), total or int(n))
    return float("inf") if n is None else f * n / fs


def window_note(n):
    return "the whole trace" if n is None else f"the first {int(n)} samples"


def component_curve(f, amp, ph, n_cycles=None, n_pts=800, span=None, t0_samples=0):
    """The fitted sinusoid on a fine grid, so a 200 Hz component is not drawn from eight samples.

    `span` is a length in samples: the component is then drawn over the same stretch of absolute
    time as the signal it was fitted on, which is what puts every panel of the page on one axis.
    """
    if COMPONENT_WINDOW == "signal" and span:
        t = np.linspace(0, span / FS, n_pts)
    else:
        n_cycles = COMPONENT_CYCLES if n_cycles is None else n_cycles
        t = np.linspace(0, n_cycles / max(f, 1e-9), n_pts)
    y = amp * np.cos(2 * np.pi * f * t - ph)
    return (t + t0_samples / FS) * 1000.0, y


def params_caption(width=118):
    """Three short lines: what generated the input, what it is made of, and what predicted it."""
    import textwrap
    if FAMILY == "tsmixup":
        pool_now = source_pool(verbose=False)
        band_txt = f"{min(pool_now):g}-{max(pool_now):g}"
        drawn = [f for f in REF_FREQS if f != INJECTED_LOCK]
        w_of = dict(zip(sorted(drawn), [WEIGHTS[i] for i in np.argsort(np.argsort(drawn))]))
        head = (f"Input : Light TSMixup, K={len(drawn)}, alpha={ALPHA_DIR}, "
                f"drawn from {band_txt} Hz, seed {DRAW_SEED}"
                + (f", planted lock at {INJECTED_LOCK:g} Hz "
                   f"({INJECT_LOCK_REL:g}x)" if INJECTED_LOCK else ""))
        body = " | ".join(
            f"{f:.2f} Hz " + (f"(w={w_of[f]:.2f})" if f in w_of else "(planted)")
            + (" [lock]" if is_lock(f) else "") for f in REF_FREQS)
    else:
        head = (f"Input : KernelSynth, J={KS_PARAMS.get('J')}, l_syn={KS_PARAMS.get('l_syn')}, "
                f"jitter={KS_PARAMS.get('jitter')}, seed {DRAW_SEED}"
                + (f", planted lock at {INJECTED_LOCK:g} Hz" if INJECTED_LOCK else ""))
        body = "kernels: " + " ".join(KERNELS)
    model = (f"Model : {MODEL_LABEL}, prediction {len(OUTPUT)} samples "
             f"({FS / len(OUTPUT):.0f} Hz bins), fit on {window_note(FIT_WINDOW_PRED)}")
    body = textwrap.fill(body, width=width, subsequent_indent="        ")
    return f"{head}\n        {body}\n{model}"


def add_params_strip(fig, y=0.012):
    """The draw's own parameters, printed on the figure so it can be read without the notebook."""
    fig.text(0.045, y, params_caption(), ha="left", va="bottom", fontsize=7.0, color="0.3",
             family="monospace", linespacing=1.5)


def draw_signal(ax, y, color, title, fs=FS, t0_samples=0):
    """One signal against absolute time, so a panel's width means the same thing on both sides."""
    t = (np.arange(len(y)) + t0_samples) / fs * 1000.0
    ax.plot(t, y, color=color, lw=0.8)
    ax.set_title(f"{title}\n{len(y)} samples, {len(y) / fs * 1000:.0f} ms", fontsize=9, pad=4)
    ax.set_xlabel("time [ms]", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.margins(x=0.01)


def panel_widths(base_l=1.45, base_r=1.45, floor=0.32):
    """Widths for the two signal columns: in proportion to how much time each one covers.

    The prediction is a horizon, not a second recording: it spans a fraction of the context, and
    equal-width panels would silently stretch it to look as long. The floor keeps a very short
    horizon legible.
    """
    if not MATCH_TIME_SCALE:
        return base_l, base_r
    ratio = float(np.clip(len(OUTPUT) / max(len(REAL), 1), floor, 1.0))
    return base_l, base_r * ratio


def page_ylim(amps=()):
    """The half-height every component panel of a page is drawn to.

    One scale for the page, set by the strongest component of the real signal: autoscaling each
    panel makes a component at one per cent of the input fill its box and read as a line.
    """
    ref = max([a for a, _ in components_of(REAL, REF_FREQS, n=FIT_WINDOW_REAL)] + [1e-9])
    if AMP_SCALE == "own":
        return 1.15 * max(list(amps) + [1e-9])
    return 1.15 * max([ref] + list(amps))       # never clip a component that exceeds the scale


def amp_ticks(ax, ylim, frac=1.15, mark=None):
    """Ticks for the panel's scale, plus the amplitude it actually holds when the two differ.

    With one scale shared across a page every panel would otherwise carry the same three numbers,
    which says what the axis is but not what the component is.
    """
    a = ylim / frac
    ticks = [-a, 0.0, a]
    if mark and 0.12 * a < mark < 0.93 * a:
        ticks += [-mark, mark]
    ticks = sorted(set(round(t, 6) for t in ticks))
    ax.set_yticks(ticks)
    ax.set_yticklabels(["0" if abs(t) < 1e-9 else f"{t:.3g}" for t in ticks], fontsize=5)
    if mark and 0.12 * a < mark < 0.93 * a:
        for v in (-mark, mark):
            ax.axhline(v, color="0.6", lw=0.5, ls=":", zorder=1)
    ax.set_ylabel("amplitude", fontsize=5.5)


def arrow(fig, ax_from, ax_to, side="right"):
    """An arrow from the edge of a signal panel to the edge of a component panel."""
    if side == "right":
        a, b = (1.005, 0.5), (-0.05, 0.5)
    else:
        a, b = (-0.005, 0.5), (1.05, 0.5)
    fig.add_artist(ConnectionPatch(xyA=a, coordsA=ax_from.transAxes,
                                   xyB=b, coordsB=ax_to.transAxes,
                                   arrowstyle="-|>", mutation_scale=11, lw=0.8, color="0.45"))

## 4, The main components of each signal, side by side

Each signal is taken apart into **its own** main components: the real signal into the frequencies
it was built from, the prediction into its own strongest lines, in separate panels rather than
overlaid. Read across a row and the two are unrelated; read down a column and you see what each
signal is made of. An arrow ties every panel to the signal it came from, and a component planted
on a lock frequency, when one was planted, is drawn in purple on whichever side it appears.


In [9]:
def own_components_page(fname):
    """Real signal and its components on the left, prediction and its components on the right."""
    freqs_l = list(REF_FREQS)
    freqs_r = prediction_lines()
    all_lines = top_peaks(OUTPUT)
    amps_all = [a for a, _ in components_of(OUTPUT, all_lines, n=FIT_WINDOW_PRED)]
    print(f"    figure 3 keeps {len(freqs_r)} of {len(all_lines)} output lines, floor "
          f"{amplitude_floor(amps_all):.4f} = {MIN_AMP_REL:g} x the strongest {MIN_AMP_REF}")
    k = max(len(freqs_l), len(freqs_r))

    fit_l = components_of(REAL, freqs_l, n=FIT_WINDOW_REAL)
    fit_r = components_of(OUTPUT, freqs_r, n=FIT_WINDOW_PRED)
    # each column on its own scale here: the left one set by the real signal, the right one by
    # the prediction, so a quiet prediction is still legible instead of being flattened
    ylim_l = page_ylim([a for a, _ in fit_l])
    ylim_r = 1.15 * max([a for a, _ in fit_r] + [1e-9])

    w_l, w_r = panel_widths()
    extra = 0.5 if INJECTED_LOCK is not None else 0.0   # room for the planted-component legend
    H = 2.3 * k + 3.8 + extra    # the headings run to three lines, hence the extra room on top
    fig = plt.figure(figsize=(13.0, H))
    gs = GridSpec(k, 4, figure=fig, width_ratios=[w_l, 1.0, 1.0, w_r],
                  wspace=0.28, hspace=0.62, left=0.045, right=0.985,
                  top=1 - 1.75 / H, bottom=(1.15 + extra) / H)
    ax_l, ax_r = fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[:, 3])
    draw_signal(ax_l, REAL, C_IN, f"{NAME} signal (real, {REAL_SEGMENT})",
                t0_samples=REAL_T0)
    draw_signal(ax_r, OUTPUT, C_OUT, "Chronos prediction", t0_samples=CTX)

    rows = []
    for i in range(k):
        if i < len(freqs_l):
            f, (amp, ph) = freqs_l[i], fit_l[i]
            planted = is_planted(f)
            a1 = fig.add_subplot(gs[i, 1])
            draw_one(a1, f, amp, ph, C_PLANT if planted else C_IN, ylim_l,
                     note="  <- planted" if planted else "",
                     span=len(REAL), t0_samples=REAL_T0)
            arrow(fig, ax_l, a1, "right")
            rows.append(dict(side="real", freq_hz=f, is_lock=is_lock(f), planted=planted,
                             amp=amp))
        if i < len(freqs_r):
            f, (amp, ph) = freqs_r[i], fit_r[i]
            c = cycles_in(f, FIT_WINDOW_PRED, total=len(OUTPUT))
            note = f"  [{c:.1f} cyc]" if c < MIN_CYCLES_WARN else ""
            g = near_input(f)
            if g is not None and abs(g - f) > 1e-9:
                note += f"  = input {g:.2f}"
            if above_pool(f):
                note += "  > pool limit"
            # one bin of tolerance: the planted site cannot be located more finely than that
            planted = is_planted(f, tol=resolution_hz(OUTPUT))
            if planted:
                note += "  <- planted"
            a2 = fig.add_subplot(gs[i, 2])
            draw_one(a2, f, amp, ph, C_PLANT if planted else C_OUT, ylim_r, note=note,
                     span=len(OUTPUT), t0_samples=CTX)
            arrow(fig, ax_r, a2, "left")
            rows.append(dict(side="prediction", freq_hz=f, is_lock=is_lock(f), planted=planted,
                             amp=amp))

    fig.suptitle(f"{NAME}: the main components of the real signal and of the prediction",
                 fontsize=12, y=1 - 0.28 / H)
    fig.text(0.5, 1 - 0.72 / H,
             f"Left column, the frequencies the real signal is built from; right column, the "
             f"strongest lines of the prediction.\nThe two are separate decompositions and a row "
             f"does not pair them: what matters is which frequencies appear, and how strongly.\nEach "
             f"column carries its own vertical scale, the left one set by the real signal and the "
             f"right one by the prediction, so the amplitudes must be read off the axes."
             + ("\nThe planted component, and the predicted line that falls on it, are drawn in "
                "purple." if INJECTED_LOCK is not None else ""),
             ha="center", va="top", fontsize=8.5, color="0.25")
    add_params_strip(fig, y=0.10 / H)
    if INJECTED_LOCK is not None:
        fig.legend(handles=[
            Line2D([0], [0], color=C_IN, lw=1.6, label="component of the real signal"),
            Line2D([0], [0], color=C_OUT, lw=1.6, label="line of the prediction"),
            Line2D([0], [0], color=C_PLANT, lw=2.0,
                   label=f"planted on {INJECTED_LOCK:g} Hz, a lock of p{REF_P}-s{REF_S}, at "
                         f"{INJECT_LOCK_REL:g}x the strongest component")],
            loc="lower center", ncol=3, fontsize=8, frameon=False,
            bbox_to_anchor=(0.5, 0.66 / H))
    stamp_stub(fig)
    p = OUT / "figures" / fname
    fig.savefig(p, dpi=200, bbox_inches="tight")
    fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig); report_written(p); report_written(p.with_suffix(".pdf"))
    return p, pd.DataFrame(rows)


def draw_one(ax, f, amp, ph, colour, ylim, n_cycles=None, note="", span=None, t0_samples=0):
    """One component, in its own axes, over the window of the signal it was fitted on."""
    t_ms, y = component_curve(f, amp, ph, n_cycles, span=span, t0_samples=t0_samples)
    ax.axhline(0, color="0.85", lw=0.6, zorder=0)
    ax.plot(t_ms, y, color=colour, lw=1.3)
    ax.set_ylim(-ylim, ylim)
    ax.set_title(f"{f:.2f} Hz{'  (lock)' if is_lock(f) else ''}{note}\nA = {amp:.3f}",
                 fontsize=7, pad=2)
    ax.set_xlabel("time [ms]", fontsize=6)
    ax.tick_params(labelsize=5)
    amp_ticks(ax, ylim, mark=amp)
    ax.margins(x=0.01)


page_c, table_c = own_components_page(f"FIG_E3_{run_suffix()}_main_components.png")
table_c.to_csv(OUT / f"components_{run_suffix()}.csv", index=False)
display(table_c.round(4))

    1 prediction line(s) below A = 0.1269 (0.45 x the strongest prediction component) dropped
    kept: 8.00 Hz (A=0.282), 16.00 Hz (A=0.129)
    figure 3 keeps 2 of 3 output lines, floor 0.1269 = 0.45 x the strongest prediction
    wrote FIG_E3_kernelsynth_published_main_components.png  283 kB  12:03:46  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures
    wrote FIG_E3_kernelsynth_published_main_components.pdf  40 kB  12:03:46  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures


,side,freq_hz,is_lock,planted,amp
0,real,2.8235,False,False,0.8775
1,prediction,8.0000,False,False,0.2821
2,prediction,16.0000,False,False,0.1292


## 5, The spectra themselves

The page above shows components one by one, which says what was looked for but not what is there.
This one shows the two magnitude spectra with the lines the search returned marked on them, so the
choice of frequencies can be checked against the transform rather than taken on trust.


In [10]:
def draw_spectrum(ax, x, peaks, colour, title, marks=None, marks_label=None, planted=None):
    fr, mag = spectrum(x)
    ax.plot(fr, mag, color=colour, lw=1.0, marker="o", ms=2.2, mew=0)
    for f in LOCKS_REF:
        ax.axvline(f, color="k", ls=(0, (4, 3)), lw=0.7, alpha=0.55, zorder=0)
    if marks:
        for i, f in enumerate(marks):
            ax.axvline(f, color="#2ca02c", lw=1.0, alpha=0.8, zorder=1,
                       label=marks_label if i == 0 else None)
    if planted is not None:                 # the component added before the model saw the signal
        ax.axvline(planted, color=C_PLANT, lw=1.8, alpha=0.95, zorder=2)
        ax.annotate(f"planted {planted:g} Hz", xy=(planted, 1.0), xycoords=("data", "axes fraction"),
                    xytext=(3, -10), textcoords="offset points", ha="left", va="top",
                    fontsize=6.5, color=C_PLANT, rotation=90)
    for f in peaks:
        j = int(np.argmin(np.abs(fr - f)))
        ax.plot([fr[j]], [mag[j]], marker="v", color="k", ms=5, mew=0, zorder=4)
        ax.annotate(f"{f:.1f}", (fr[j], mag[j]), textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=6.5)
    ax.set_xlim(0, FS / 2)
    ax.set_title(title, fontsize=9.5, pad=4)
    ax.set_xlabel("frequency [Hz]", fontsize=7.5)
    ax.set_ylabel("magnitude", fontsize=7.5)
    ax.tick_params(labelsize=6.5)
    ax.axvspan(0, BAND[0], color="0.9", lw=0, zorder=0)
    ax.axvspan(BAND[1], FS / 2, color="0.9", lw=0, zorder=0)
    cap = pool_limit()
    if cap is not None and cap < BAND[1]:       # nothing the generator could have put here
        ax.axvspan(cap, BAND[1], color="#f2c14e", alpha=0.16, lw=0, zorder=0)
        ax.axvline(cap, color="#b8860b", lw=1.0, ls="-.", alpha=0.9, zorder=1)


def spectra_page(fname):
    """The two magnitude spectra, with the detected lines, the comb and the amplitude cut."""
    peaks_r = prediction_lines()
    cand = top_peaks(OUTPUT)
    floor = amplitude_floor([a for a, _ in components_of(OUTPUT, cand, n=FIT_WINDOW_PRED)])
    H = 7.8
    fig = plt.figure(figsize=(12.0, H))
    gs = GridSpec(2, 1, figure=fig, hspace=0.42, left=0.07, right=0.985, top=0.85,
                  bottom=1.05 / H)
    ax1, ax2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    drawn = [f for f in REF_FREQS if not is_planted(f)]      # the planted one is not a draw
    draw_spectrum(ax1, REAL, REF_FREQS, C_IN,
                  f"{NAME} signal (real, {REAL_SEGMENT}), {len(REAL)} samples, "
                  f"resolving power {FS / len(REAL):.1f} Hz",
                  marks=(drawn if FAMILY == "tsmixup" else None),
                  marks_label="frequency drawn by the generator",
                  planted=INJECTED_LOCK)
    draw_spectrum(ax2, OUTPUT, peaks_r, C_OUT,
                  f"Chronos prediction, {len(OUTPUT)} samples, "
                  f"resolving power {FS / len(OUTPUT):.1f} Hz",
                  planted=INJECTED_LOCK)
    ax2.axhline(floor, color="#8c1010", lw=1.2, ls="--", zorder=6)
    ax2.annotate(f"cut at A = {floor:.4f}", xy=(BAND[1], floor), xytext=(-4, 3),
                 textcoords="offset points", ha="right", va="bottom",
                 fontsize=7.5, color="#8c1010", zorder=6)
    for ax in (ax1, ax2):
        h, l = ax.get_legend_handles_labels()
        handles = [Line2D([0], [0], color="k", ls=(0, (4, 3)), lw=1.0,
                          label=r"$f\in\mathcal{F}_{lock}$ of the geometry"),
                   Line2D([0], [0], marker="v", color="k", lw=0, label="line the search returned")]
        cap = pool_limit()
        if cap is not None and cap < BAND[1]:
            handles.append(Line2D([0], [0], color="#b8860b", ls="-.", lw=1.2,
                                  label=f"top of the source pool, {cap:g} Hz"))
        if INJECTED_LOCK is not None:
            handles.append(Line2D([0], [0], color=C_PLANT, lw=1.8,
                                  label=f"planted component, {INJECTED_LOCK:g} Hz "
                                        f"({INJECT_LOCK_REL:g}x)"))
        if ax is ax2:
            handles.append(Line2D([0], [0], color="#8c1010", ls="--", lw=1.2,
                                  label=f"cut, {100 * MIN_AMP_REL:g}% of the "
                                        f"{MIN_AMP_REF.replace('_', ' ')}"))
        ax.legend(handles=handles + h, fontsize=7, ncol=3, loc="upper right", framealpha=0.85)
    fig.suptitle(f"{NAME}: the transforms the lines were read off", fontsize=12, y=0.965)
    fig.text(0.5, 0.905,
             f"Hann window, one-sided, scaled so a pure tone of amplitude A peaks near A; grey "
             f"bands lie outside the analysed range {BAND[0]:g}-{BAND[1]:g} Hz.\nEvaluated on a "
             f"{SPECTRUM_BIN_HZ:g} Hz grid by zero padding: that places a peak finely, but two "
             f"lines closer than {FS / len(OUTPUT):.0f} Hz still merge into one.",
             ha="center", fontsize=8.5, color="0.25")
    add_params_strip(fig, y=0.10 / H)
    stamp_stub(fig)
    p = OUT / "figures" / fname
    fig.savefig(p, dpi=200, bbox_inches="tight")
    fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig); report_written(p); report_written(p.with_suffix(".pdf"))
    return p


page_d = spectra_page(f"FIG_E4_{run_suffix()}_spectra.png")

    1 prediction line(s) below A = 0.1269 (0.45 x the strongest prediction component) dropped
    kept: 8.00 Hz (A=0.282), 16.00 Hz (A=0.129)
    wrote FIG_E4_kernelsynth_published_spectra.png  235 kB  12:03:47  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures
    wrote FIG_E4_kernelsynth_published_spectra.pdf  47 kB  12:03:48  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures


## 6, The other family, on the same model

Figures 3 and 4 are wanted for both generators, and the model is already loaded. This section
draws the family the run did not start with, sends it through the same checkpoint and writes its
two pages: same geometry, same settings, same code, one more draw and one forward pass. Names
carry the family, so nothing here overwrites what section 4 and section 5 produced.

The globals are left holding the last family run. To go back, re-run sections 1 and 2.


In [11]:
OTHER = [f for f in ("tsmixup", "kernelsynth") if f != FAMILY]   # set to [] to skip

for fam in OTHER:
    try:                       # one family failing must not cost the other its pages
        build_signal(fam)
        describe_draw(show=False)
        build_output()
        _p3, _t3 = own_components_page(f"FIG_E3_{run_suffix()}_main_components.png")
        _t3.to_csv(OUT / f"components_{run_suffix()}.csv", index=False)
        _p4 = spectra_page(f"FIG_E4_{run_suffix()}_spectra.png")
        display(Markdown(f"**{NAME}**"), _t3.round(4))
    except Exception as e:
        print(f"{fam} failed: {type(e).__name__}: {e}")
        if isinstance(e, ModuleNotFoundError):
            print("   KernelSynth needs chronos/data/synthetic/signalGenerator.py on the path; "
                  "probe_lib adds that directory when it imports, so a checkout without the file "
                  "is the usual cause")

print("globals now hold:", NAME, "|", run_suffix())

--- Light TSMixup, seed 50 ----------------------------------------
pool: 397 frequencies from 2 to 150 Hz (probe_lib.tsmixup_pool; probe_lib defines the cap)
drew 8 components (K = 8)
drawn components [Hz]: [12.366, 37.333, 59.721, 83.0, 97.0, 115.756, 134.0, 146.0]
mixing weights      : [0.088 0.058 0.294 0.048 0.142 0.157 0.056 0.156]
signal 544 samples, context 480, std 1.000
settings:
             generator = Light TSMixup
                     K = 8
              k_random = False
       planted_lock_hz = None
                 alpha = 1.5
             pool_size = 397
     min_separation_hz = 10.0
         draw_range_hz = (2.0, 250.0)
                length = 544
                 fs_hz = 512
                  seed = 50


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

model: amazon/chronos-bolt-tiny  (P=16, S=16)
    every file written from here on is named  *_tsmixup_published.*
real segment: 'future', 64 samples starting at 938 ms
context 480 samples (938 ms), prediction 64 samples (125 ms), std 0.3801
spectra evaluated on a 1 Hz grid (zero padded)
resolving power: prediction 8.0 Hz, real segment 8.0 Hz -- two lines closer than that are one line, not two
    5 prediction line(s) below A = 0.2196 (0.45 x the strongest prediction component) dropped
    kept: 40.00 Hz (A=0.488)
    figure 3 keeps 1 of 6 output lines, floor 0.2176 = 0.45 x the strongest prediction
    wrote FIG_E3_tsmixup_published_main_components.png  911 kB  12:03:52  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures
    wrote FIG_E3_tsmixup_published_main_components.pdf  59 kB  12:03:53  -> /content/patchAliasing/chronos/bayesian/_run/appendixE/figures
    5 prediction line(s) below A = 0.2196 (0.45 x the strongest prediction component) dropped
    kept: 40.00 Hz (A

## 7, The files this run wrote

Every page is written twice, PNG and PDF, under a name carrying the family, the model that
actually loaded and the planted component when there is one. What goes into the report is a row
marked *this run*: files from earlier runs sit in the same folder under different names and are
easy to pick up by mistake.


In [12]:
import time

_rows = []
for f in sorted((OUT / "figures").glob("*")) + sorted(OUT.glob("*.csv")):
    st = f.stat()
    _rows.append(dict(file=f.name, kB=round(st.st_size / 1024),
                      modified=time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(st.st_mtime)),
                      family=next((k for k in ("tsmixup", "kernelsynth") if k in f.name), ""),
                      model=("published" if "published" in f.name else
                             "stub" if "stub" in f.name else "retrained"),
                      planted=("yes" if "_lock" in f.name else "")))
FILES = pd.DataFrame(_rows).sort_values("modified", ascending=False)
print(f"{OUT}")
print("figures of this session carry the tags:",
      ", ".join(sorted({r["file"].split("FIG_E")[-1][2:].rsplit("_", 2)[0]
                        for r in _rows if r["file"].startswith("FIG_E")})) or "none yet")
display(FILES)

/content/patchAliasing/chronos/bayesian/_run/appendixE
figures of this session carry the tags: kernelsynth, kernelsynth_p16-s16, kernelsynth_published, tsmixup, tsmixup_p16-s16, tsmixup_published


,file,kB,modified,family,model,planted
15,FIG_E4_tsmixup_published_spectra.png,299,2026-08-26 12:03:54,tsmixup,published,
14,FIG_E4_tsmixup_published_spectra.pdf,51,2026-08-26 12:03:54,tsmixup,published,
19,components_tsmixup_published.csv,0,2026-08-26 12:03:53,tsmixup,published,
6,FIG_E3_tsmixup_published_main_components.pdf,59,2026-08-26 12:03:53,tsmixup,published,
7,FIG_E3_tsmixup_published_main_components.png,911,2026-08-26 12:03:52,tsmixup,published,
23,generator_params_tsmixup_published.csv,0,2026-08-26 12:03:48,tsmixup,published,
10,FIG_E4_kernelsynth_published_spectra.pdf,47,2026-08-26 12:03:48,kernelsynth,published,
11,FIG_E4_kernelsynth_published_spectra.png,235,2026-08-26 12:03:47,kernelsynth,published,
2,FIG_E3_kernelsynth_published_main_components.pdf,40,2026-08-26 12:03:46,kernelsynth,published,
3,FIG_E3_kernelsynth_published_main_components.png,283,2026-08-26 12:03:46,kernelsynth,published,


In [13]:
# Saving the figures somewhere that outlives the runtime.  On Colab `_run/` lives on the VM's own
# disk and disappears when the session ends, so the figures are mirrored to Drive; elsewhere the
# same call copies them to DRIVE_DIR if that path exists, and is otherwise a no-op.
import importlib.util, shutil

DRIVE_COPY = True
DRIVE_DIR  = "/content/drive/MyDrive/appendixE_figures"


def mirror_to_drive(src=None, dest=DRIVE_DIR, patterns=("*.png", "*.pdf", "*.csv")):
    """Copy the figures out of `_run/` and into Drive.  Returns the destination, or None."""
    if not DRIVE_COPY:
        print("DRIVE_COPY is off; figures stay under", OUT)
        return None
    src = Path(src or (OUT / "figures"))
    on_colab = importlib.util.find_spec("google.colab") is not None
    if on_colab and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")                 # asks once per runtime
    dest = Path(dest)
    if not on_colab and not dest.parent.exists():
        print(f"not on Colab and {dest.parent} does not exist; figures stay under {src}")
        return None
    dest.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in patterns:
        for f in sorted(list(src.glob(pattern)) + list(src.parent.glob(pattern))):
            shutil.copy2(f, dest / f.name)
            copied.append(f.name)
    print(f"copied {len(copied)} files -> {dest}")
    for name in copied:
        print("   ", name)
    return dest


mirror_to_drive()

copied 24 files -> /content/drive/MyDrive/appendixE_figures
    FIG_E3_kernelsynth_p16-s16_main_components.png
    FIG_E3_kernelsynth_published_main_components.png
    FIG_E3_tsmixup_p16-s16_main_components.png
    FIG_E3_tsmixup_published_main_components.png
    FIG_E4_kernelsynth_p16-s16_spectra.png
    FIG_E4_kernelsynth_published_spectra.png
    FIG_E4_tsmixup_p16-s16_spectra.png
    FIG_E4_tsmixup_published_spectra.png
    FIG_E3_kernelsynth_p16-s16_main_components.pdf
    FIG_E3_kernelsynth_published_main_components.pdf
    FIG_E3_tsmixup_p16-s16_main_components.pdf
    FIG_E3_tsmixup_published_main_components.pdf
    FIG_E4_kernelsynth_p16-s16_spectra.pdf
    FIG_E4_kernelsynth_published_spectra.pdf
    FIG_E4_tsmixup_p16-s16_spectra.pdf
    FIG_E4_tsmixup_published_spectra.pdf
    components_kernelsynth_p16-s16.csv
    components_kernelsynth_published.csv
    components_tsmixup_p16-s16.csv
    components_tsmixup_published.csv
    generator_params_kernelsynth_p16-s16.csv
    gen

PosixPath('/content/drive/MyDrive/appendixE_figures')

## 8, What goes in the appendix, and what may be claimed

Two `figure*` pages per family and model, each `width=\textwidth`. The caption carries the reference
geometry, how the output was produced (rollout or single horizon) and the resolution that follows
from it, and the fact that amplitudes are least squares at a known frequency rather than DFT bins.

Three limits belong in the text, not the caption, and each is a property of the measurement rather
than of the model. The output is a median quantile, and a conditional median is smoother than a
sample path, so part of any high-frequency attenuation belongs to the point forecast. The rollout
is imposed from outside and compounds its own error, so a line that decays across it is not by
itself a statement about tokenisation. And a candidate frequency and its controls sit a few hertz
apart, which no spectrum computed here separates: the pages show loss, not the paired contrast the
Bayesian models estimate.
